In [1]:
# import requests
import pandas as pd
import numpy as np
from scipy.stats import entropy
import scipy.stats as stats

# import matplotlib.pyplot as plt
# import seaborn as sns
from statsmodels.tsa.stattools import adfuller  
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import root_mean_squared_error , mean_absolute_error , mean_absolute_percentage_error          
from sklearn.model_selection import TimeSeriesSplit


import itertools

import time
# import yaml
pd.options.display.float_format = '{:,.2f}'.format
import warnings
warnings.filterwarnings('ignore')  # Suppress all warnings

## 1. Data Preparation

In [17]:
df = pd.read_csv("italy_unemployment_rate_PREDICTION_DATASET.csv")

In [18]:
df

,date,month,employment_rate_diff,industry_production_prices_diff,companies_trust,rolling_mean_6m,rolling_mean_12m,PCA_labour_market_1,PCA_labour_market_2,PCA_price_energy_1,PCA_price_energy_2,unemployment_rate_diff
0,2014-02-01,2,1.12,0.00,87.30,0.50,0.07,1.01,0.01,0.06,0.13,-0.02
1,2014-03-01,3,0.17,-0.20,89.20,0.15,0.04,1.32,0.01,0.04,0.10,-0.73
2,2014-04-01,4,-1.24,-0.20,88.30,-0.00,0.03,-2.75,0.00,0.03,0.08,-0.30
3,2014-05-01,5,2.21,-0.10,87.60,-0.07,0.04,1.92,0.01,0.05,0.07,-0.12
4,2014-06-01,6,-0.40,0.20,89.30,-0.16,-0.03,-1.47,-0.01,0.04,0.07,-1.50
...,...,...,...,...,...,...,...,...,...,...,...,...
151,2026-09-01,9,0.04,0.13,96.27,0.02,-0.17,0.42,0.00,0.02,-0.01,NaN
152,2026-10-01,10,0.04,0.49,96.15,-0.02,-0.09,0.19,0.00,-0.01,0.01,NaN
153,2026-11-01,11,0.04,0.04,95.81,-0.02,-0.11,-0.36,0.00,0.00,0.00,NaN
154,2026-12-01,12,0.04,0.95,95.33,-0.07,-0.09,0.11,-0.00,-0.00,0.00,NaN


In [19]:
# df['date'] = pd.to_datetime(df['Unnamed: 0'])
X_columns = df.set_index('date')
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date')

In [20]:
df.index.name = None

In [21]:
df_target = df[['unemployment_rate_diff']]
df.drop('unemployment_rate_diff', axis = 1, inplace = True)

In [22]:
df_target.dropna(axis= 0, inplace =True)
df_target

,unemployment_rate_diff
2014-02-01,-0.02
2014-03-01,-0.73
2014-04-01,-0.30
2014-05-01,-0.12
2014-06-01,-1.50
...,...
2025-09-01,0.72
2025-10-01,-0.40
2025-11-01,-0.12
2025-12-01,0.31


In [23]:
selected_cols = ['month', 
                          'employment_rate_diff', 
                         'industry_production_prices_diff',
                         'companies_trust', 
                        # 'rolling_mean_3m', 
                        'rolling_mean_6m',
                        'rolling_mean_12m', 
                        # 'rolling_std_12m', 
                        'PCA_labour_market_1',
                        'PCA_labour_market_2', 
                        'PCA_price_energy_1', 
                         'PCA_price_energy_2'
                        ]  

In [26]:
X_train = df.iloc[:-12]
X_test = df.iloc[-12:]

y_train = df_target.iloc[:-12]['unemployment_rate_diff']
y_test = df_target.iloc[-12:]['unemployment_rate_diff']

## 2. Model: SARIMAX

Riparti da qui§

In [21]:
best_param_V1 = (1, 1, 0)
best_param_seasonal_V1 = (0, 1, 0, 12)

# best_param_V2 = (0, 2, 2)
# best_param_seasonal_V2 = (0, 2, 2, 12)

### 2.1 training

In [ ]:
model = SARIMAX(y_train,
                exog=exog_train[selected_cols], 
                order=best_param,
                seasonal_order=best_param_seasonal,
                enforce_stationarity=False,
                enforce_invertibility=False)
results = model.fit()


y_val_pred = results.predict(
    start=exog_val.index[0],
    end=exog_val.index[-1],
    exog=exog_val[selected_cols]
)


#################################################################
######################## TESTING  ###############################
#################################################################
exog_future = pd.concat([exog_val,  exog_test])

y_test_pred = results.predict(
        start=exog_test.index[0],
        end=exog_test.index[-1],
        exog=exog_future[selected_cols]
            )

In [22]:
model = SARIMAX(y_train,
                exog=X_train, 
                order=best_param_V1,
                seasonal_order=best_param_seasonal_V1,
                enforce_stationarity=False,
                enforce_invertibility=False)
results = model.fit()


h = 12  # forecast horizon

# Training
exog_train_pred = X_train.iloc[-h:]  # 121 months
 

forecast_res = results.get_forecast(
    steps=h,
    exog=exog_train_pred              # required if you used exog
)

y_pred_forecast_mean_train = forecast_res.predicted_mean
conf_int = forecast_res.conf_int(alpha=0.05)  # 95% CI

In [23]:
y_train[-12:]

date
2025-02-01   183,989,804.00
2025-03-01   198,276,950.00
2025-04-01   204,787,548.00
2025-05-01   205,933,595.00
2025-06-01   211,838,742.00
2025-07-01   223,889,631.00
2025-08-01   159,704,179.00
2025-09-01   207,871,871.00
2025-10-01   220,481,612.00
2025-11-01   219,885,862.00
2025-12-01   222,564,373.00
2026-01-01   202,235,203.00
Name: TotSales, dtype: float64

In [24]:
y_pred_forecast_mean_train

2026-02-01   190,555,738.63
2026-03-01   204,713,916.20
2026-04-01   211,245,175.35
2026-05-01   212,387,912.36
2026-06-01   218,293,589.63
2026-07-01   230,344,393.68
2026-08-01   166,158,955.29
2026-09-01   214,326,645.11
2026-10-01   226,936,386.46
2026-11-01   226,340,636.41
2026-12-01   229,019,147.41
2027-01-01   208,689,977.41
Freq: MS, Name: predicted_mean, dtype: float64

### 2.2 testing

In [25]:
#THE OLD ONE

h = 12  # forecast horizon

forecast_res_test = results.get_forecast(
                                            steps=h,
                                            exog=X_test      # required if you used exog
                                        )

y_pred_forecast_mean_test = forecast_res_test.predicted_mean
conf_int = forecast_res_test.conf_int(alpha=0.05)  # 95% CI

In [26]:
y_original = df['TotSales'].iloc[-13] 

In [27]:
# Reversing the difference:
anchor = df['TotSales'].iloc[-13] 
final_forecast = y_pred_forecast_mean_test.cumsum() + anchor

In [28]:
final_forecast

2026-02-01     384,123,276.09
2026-03-01     590,399,695.15
2026-04-01     787,425,378.03
2026-05-01     982,135,855.18
2026-06-01   1,190,080,611.45
2026-07-01   1,403,111,658.12
2026-08-01   1,554,116,934.88
2026-09-01   1,754,925,881.97
2026-10-01   1,953,432,307.35
2026-11-01   2,160,469,091.22
2026-12-01   2,361,968,748.08
2027-01-01   2,540,005,897.34
Freq: MS, Name: predicted_mean, dtype: float64

In [29]:
y_pred_forecast_mean_test

2026-02-01   181,888,073.09
2026-03-01   206,276,419.06
2026-04-01   197,025,682.88
2026-05-01   194,710,477.16
2026-06-01   207,944,756.27
2026-07-01   213,031,046.67
2026-08-01   151,005,276.76
2026-09-01   200,808,947.08
2026-10-01   198,506,425.38
2026-11-01   207,036,783.87
2026-12-01   201,499,656.86
2027-01-01   178,037,149.26
Freq: MS, Name: predicted_mean, dtype: float64

In [39]:
df['TotSales'] = df.TotSales.fillna(y_pred_forecast_mean_test)

df['date'] = df.index

df = df[['date', 'WorkingDays', 'fixed_term', 'employment_rate', 'youth_employment_rate',
                'industry_production_prices', 'diesel_price', 'TotSales']]

In [40]:
df

,date,WorkingDays,fixed_term,employment_rate,youth_employment_rate,industry_production_prices,diesel_price,TotSales
date,,,,,,,,
2014-02-01,2014-02-01,20,"2,116,627.00",38.64,54.49,93.00,1.64,"78,038,658.00"
2014-03-01,2014-03-01,21,"2,188,435.00",38.81,55.42,92.80,1.63,"84,094,082.00"
2014-04-01,2014-04-01,20,"2,215,274.00",37.57,54.69,92.60,1.63,"86,341,669.00"
2014-05-01,2014-05-01,21,"2,255,743.00",39.78,55.44,92.50,1.63,"90,493,760.00"
2014-06-01,2014-06-01,20,"2,436,989.00",39.38,55.90,92.70,1.63,"90,851,778.00"
...,...,...,...,...,...,...,...,...
2026-09-01,2026-09-01,22,"2,714,840.53",42.35,59.79,90.85,1.62,"200,808,947.08"
2026-10-01,2026-10-01,21,"2,714,840.53",42.14,59.57,87.28,1.58,"198,506,425.38"
2026-11-01,2026-11-01,22,"2,714,840.53",42.01,59.13,86.18,1.57,"207,036,783.87"


In [41]:
df.to_excel('Italy_revenues_predicted_2026.xlsx', index=False)